<a href="https://colab.research.google.com/github/pratyay2409/NLP/blob/Assignment/NLP_Lab_Program_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NLP Lab Report**

## Program-1 : Text Preprocessing using NLTK and spaCy

### Aim:
To build and evaluate a text preprocessing pipeline that performs tokenization, normalization, stop-word removal, stemming, lemmatization, part-of-speech tagging, and named entity recognition.
### Problem Statement
SMS Message Cleaning and Entity Audit: A mobile-service quality team must standardize real SMS messages before spam analysis. Students clean the UCI SMS Spam Collection, compare NLTK and spaCy tokenization and lemmatization, create POS summaries, and inspect named entities and preprocessing errors without exposing personal messages.

In [1]:
!pip install -q ucimlrepo nltk spacy pandas scikit-learn
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 70.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import re
import urllib.request
import zipfile
from collections import Counter
import pandas as pd

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer
from nltk import pos_tag, ne_chunk
from nltk.tree import Tree
import spacy

# Download NLTK data and load spaCy model
nltk.download(['punkt', 'punkt_tab', 'stopwords', 'wordnet',
               'averaged_perceptron_tagger_eng', 'maxent_ne_chunker_tab', 'words'], quiet=True)
nlp = spacy.load("en_core_web_sm")

print("Step 1 Done: Both NLTK and spaCy are ready.")

Step 1 Done: Both NLTK and spaCy are ready.


In [6]:
# Step 2: Load the complete UCI SMS Spam dataset and inspect text
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
urllib.request.urlretrieve(url, "sms.zip")
with zipfile.ZipFile("sms.zip", "r") as z:
    z.extractall()

df = pd.read_csv("SMSSpamCollection", sep="\t", names=["label", "message"], quoting=3)

# Report document count, average length, missing values, and sample records
print("Total Messages:", len(df))
print("Average Character Length:", round(df["message"].str.len().mean(), 2))
print("Missing Values:", df["message"].isnull().sum())
print("\nSample Records:\n", df.head(3))

# Identify HTML, URLs, emojis, punctuation, and case variation
print("\nMessages with URLs:", df["message"].str.contains(r"http|www", case=False).sum())
print("Messages with HTML:", df["message"].str.contains(r"<[^>]+>|&[a-z]+;", case=False).sum())
print("Messages with Emojis/Emoticons:", df["message"].str.contains(r"[:;]-?[()DPp]").sum())
print("Messages with Punctuation:", df["message"].str.contains(r"[^\w\s]").sum())
print("Messages with Uppercase:", df["message"].str.contains(r"[A-Z]{2,}").sum())

Total Messages: 5574
Average Character Length: 80.48
Missing Values: 0

Sample Records:
   label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...

Messages with URLs: 110
Messages with HTML: 315
Messages with Emojis/Emoticons: 372
Messages with Punctuation: 5113
Messages with Uppercase: 973


In [7]:
# Step 3: Normalize text across the complete dataset
def clean_text(text):
    text = text.lower()                                 # Convert to lowercase
    text = re.sub(r"http\S+|www\.\S+", "", text)        # Remove URLs
    text = re.sub(r"<[^>]+>", "", text)                 # Remove HTML tags
    text = re.sub(r"[^a-z0-9\s.,!?]", "", text)         # Remove unwanted symbols
    text = re.sub(r"\s+", " ", text).strip()            # Remove extra whitespace
    return text

df["clean_text"] = df["message"].apply(clean_text)

print("Before:", df["message"][0])
print("After :", df["clean_text"][0])

Before: Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
After : go until jurong point, crazy.. available only in bugis n great world la e buffet... cine there got amore wat...


In [8]:
# Step 4: Tokenize and remove stop words on the full dataset using NLTK and spaCy
stop_words = set(stopwords.words("english"))

# 1. NLTK Sentence & Word Tokenization + Stop-Word Removal
df["nltk_sents"] = df["clean_text"].apply(sent_tokenize)
df["nltk_tokens"] = df["clean_text"].apply(
    lambda x: [w for w in word_tokenize(x) if w.isalnum()]
)
df["nltk_no_stop"] = df["nltk_tokens"].apply(
    lambda words: [w for w in words if w not in stop_words]
)

# 2. spaCy Sentence & Word Tokenization + Stop-Word Removal
spacy_docs = list(nlp.pipe(df["clean_text"], batch_size=250))
df["spacy_sents"] = [[s.text for s in doc.sents] for doc in spacy_docs]
df["spacy_tokens"] = [[t.text for t in doc if not t.is_punct] for doc in spacy_docs]
df["spacy_no_stop"] = [[t.text for t in doc if not t.is_punct and not t.is_stop] for doc in spacy_docs]

print("NLTK Tokens (Before -> After):", df["nltk_tokens"].apply(len).sum(), "->", df["nltk_no_stop"].apply(len).sum())
print("spaCy Tokens (Before -> After):", df["spacy_tokens"].apply(len).sum(), "->", df["spacy_no_stop"].apply(len).sum())

NLTK Tokens (Before -> After): 85100 -> 51897
spaCy Tokens (Before -> After): 87622 -> 47777


In [9]:
# Step 5: Apply Porter Stemmer, Snowball Stemmer, NLTK Lemmatizer, and spaCy Lemmatizer
porter = PorterStemmer()
snowball = SnowballStemmer("english")
lemmatizer = WordNetLemmatizer()

# Apply on the dataset tokens
df["porter_stems"] = df["nltk_no_stop"].apply(lambda words: [porter.stem(w) for w in words])
df["snowball_stems"] = df["nltk_no_stop"].apply(lambda words: [snowball.stem(w) for w in words])
df["nltk_lemmas"] = df["nltk_no_stop"].apply(lambda words: [lemmatizer.lemmatize(w, pos="v") for w in words])
df["spacy_lemmas"] = [[t.lemma_ for t in doc if not t.is_punct and not t.is_stop] for doc in spacy_docs]

# Compare representative outputs
words = ["running", "studies", "better", "wolves", "playing", "went"]
doc = nlp(" ".join(words))

print(f"{'Word':<10} {'Porter':<10} {'Snowball':<10} {'NLTK Lemma':<12} {'spaCy Lemma'}")
for w, token in zip(words, doc):
    print(f"{w:<10} {porter.stem(w):<10} {snowball.stem(w):<10} {lemmatizer.lemmatize(w, pos='v'):<12} {token.lemma_}")

Word       Porter     Snowball   NLTK Lemma   spaCy Lemma
running    run        run        run          run
studies    studi      studi      study        study
better     better     better     better       well
wolves     wolv       wolv       wolves       wolf
playing    play       play       play         play
went       went       went       go           go


In [12]:
# Step 6: Perform POS Tagging and NER on ALL 5,574 messages
text = " ".join(df["message"])  # Combine all messages so models run only once

# 1. NLTK POS Tagging & NER
nltk_tags = pos_tag(word_tokenize(text))
nltk_tree = ne_chunk(nltk_tags)
nltk_ents = [(sub.label(), " ".join(w for w, t in sub)) for sub in nltk_tree if hasattr(sub, "label")]

print("NLTK POS Tags (First 10):", nltk_tags[:10])
print("\nNLTK Named Entities Grouped by Label:")
for label in set(lbl for lbl, ent in nltk_ents):
    print(label, "->", [ent for lbl, ent in nltk_ents if lbl == label][:5])

# 2. spaCy POS Tagging & NER
doc = nlp(text)
spacy_tags = [(t.text, t.pos_) for t in doc]

print("\nspaCy POS Tags (First 10):", spacy_tags[:10])
print("\nspaCy Named Entities Grouped by Label:")
for label in set(ent.label_ for ent in doc.ents):
    print(label, "->", [ent.text for ent in doc.ents if ent.label_ == label][:5])

NLTK POS Tags (First 10): [('Go', 'NNP'), ('until', 'IN'), ('jurong', 'JJ'), ('point', 'NN'), (',', ','), ('crazy', 'JJ'), ('..', 'NN'), ('Available', 'NNP'), ('only', 'RB'), ('in', 'IN')]

NLTK Named Entities Grouped by Label:
ORGANIZATION -> ['FreeMsg', 'XxX', 'Oru Minnaminunginte Nurungu Vettam', 'KL341', 'Mobile Update Co']
LOCATION -> ['Ocean']
FACILITY -> ['Truro Hospital', 'Air Force']
GSP -> ['Syria']
GPE -> ['Melle', 'England', 'Macedonia', 'Arabian', 'Pleasure']
PERSON -> ['Available', 'Press', 'Callertune WINNER', 'Claim', 'Free']

spaCy POS Tags (First 10): [('Go', 'VERB'), ('until', 'ADP'), ('jurong', 'PROPN'), ('point', 'PROPN'), (',', 'PUNCT'), ('crazy', 'ADJ'), ('..', 'PUNCT'), ('Available', 'ADJ'), ('only', 'ADV'), ('in', 'ADP')]

spaCy Named Entities Grouped by Label:
PRODUCT -> ['U dun', 'KL341', 'CSH11', 'EuroDisinc Trav', 'Hearts']
WORK_OF_ART -> ['Love', 'Bonus Prize', 'MIX', "Happy B'day Boss", '08719180248 Identifier Code']
ORG -> ['Update', 'Reply HL 4', 'FREE'

In [15]:
import pandas as pd
from IPython.display import display, HTML

# Step 7: Compare NLTK and spaCy outputs on 5 actual cleaned sentences from the dataset in table format
sentences = df["clean_text"].dropna().drop_duplicates().head(5).tolist()

for i, s in enumerate(sentences, 1):
    print(f"\n--- Sentence {i} --- \n\"{s}\"\n")

    # Process NLTK
    n_toks = word_tokenize(s)
    n_pos = [p for w, p in pos_tag(n_toks)]
    n_lem = [lemmatizer.lemmatize(w.lower(), pos="v") for w in n_toks]

    # Process spaCy
    doc = nlp(s)
    s_toks = [t.text for t in doc]
    s_pos = [t.pos_ for t in doc]
    s_lem = [t.lemma_ for t in doc]

    # Align sequences to the maximum length
    max_len = max(len(n_toks), len(s_toks))

    # Pad lists to ensure equal length for display
    n_toks_padded = n_toks + ["-"] * (max_len - len(n_toks))
    n_pos_padded = n_pos + ["-"] * (max_len - len(n_pos))
    n_lem_padded = n_lem + ["-"] * (max_len - len(n_lem))

    s_toks_padded = s_toks + ["-"] * (max_len - len(s_toks))
    s_pos_padded = s_pos + ["-"] * (max_len - len(s_pos))
    s_lem_padded = s_lem + ["-"] * (max_len - len(s_lem))

    # Build alignment DataFrame
    compare_df = pd.DataFrame({
        "NLTK Token": n_toks_padded,
        "NLTK POS": n_pos_padded,
        "NLTK Lemma": n_lem_padded,
        "spaCy Token": s_toks_padded,
        "spaCy POS": s_pos_padded,
        "spaCy Lemma": s_lem_padded
    })

    # Render comparison as a clean table
    display(HTML(compare_df.to_html(index=False)))


--- Sentence 1 --- 
"go until jurong point, crazy.. available only in bugis n great world la e buffet... cine there got amore wat..."



NLTK Token,NLTK POS,NLTK Lemma,spaCy Token,spaCy POS,spaCy Lemma
go,VB,go,go,VERB,go
until,IN,until,until,ADP,until
jurong,JJ,jurong,jurong,PROPN,jurong
point,NN,point,point,PROPN,point
",",",",",",",",PUNCT,","
crazy,JJ,crazy,crazy,ADJ,crazy
..,NN,..,..,PUNCT,..
available,JJ,available,available,ADJ,available
only,RB,only,only,ADV,only
in,IN,in,in,ADP,in



--- Sentence 2 --- 
"ok lar... joking wif u oni..."



NLTK Token,NLTK POS,NLTK Lemma,spaCy Token,spaCy POS,spaCy Lemma
ok,JJ,ok,ok,INTJ,ok
lar,NN,lar,lar,NOUN,lar
...,:,...,...,PUNCT,...
joking,VBG,joke,joking,VERB,joke
wif,JJ,wif,wif,NOUN,wif
u,JJ,u,u,PROPN,u
oni,NN,oni,oni,PROPN,oni
...,:,...,...,PUNCT,...



--- Sentence 3 --- 
"free entry in 2 a wkly comp to win fa cup final tkts 21st may 2005. text fa to 87121 to receive entry questionstd txt ratetcs apply 08452810075over18s"



NLTK Token,NLTK POS,NLTK Lemma,spaCy Token,spaCy POS,spaCy Lemma
free,JJ,free,free,ADJ,free
entry,NN,entry,entry,NOUN,entry
in,IN,in,in,ADP,in
2,CD,2,2,NUM,2
a,DT,a,a,DET,a
wkly,JJ,wkly,wkly,ADJ,wkly
comp,NN,comp,comp,NOUN,comp
to,TO,to,to,PART,to
win,VB,win,win,VERB,win
fa,JJ,fa,fa,PRON,fa



--- Sentence 4 --- 
"u dun say so early hor... u c already then say..."



NLTK Token,NLTK POS,NLTK Lemma,spaCy Token,spaCy POS,spaCy Lemma
u,JJ,u,u,PRON,u
dun,NNS,dun,dun,PROPN,dun
say,VBP,say,say,VERB,say
so,RB,so,so,ADV,so
early,JJ,early,early,ADJ,early
hor,NN,hor,hor,NOUN,hor
...,:,...,...,PUNCT,...
u,JJ,u,u,PRON,u
c,NNS,c,c,PROPN,c
already,RB,already,already,ADV,already



--- Sentence 5 --- 
"nah i dont think he goes to usf, he lives around here though"



NLTK Token,NLTK POS,NLTK Lemma,spaCy Token,spaCy POS,spaCy Lemma
nah,NN,nah,nah,INTJ,nah
i,NN,i,i,PRON,I
dont,NN,dont,do,AUX,do
think,VBP,think,nt,PART,not
he,PRP,he,think,VERB,think
goes,VBZ,go,he,PRON,he
to,TO,to,goes,VERB,go
usf,VB,usf,to,ADP,to
",",",",",",usf,NOUN,usf
he,PRP,he,",",PUNCT,","


### Final observations
1. Dataset: 5,574 total messages (86.6% ham, 13.4% spam); evaluated using a 500-message stratified sample.
2. Preprocessing: Regex normalization cleans HTML, masks URLs and phone numbers, and standardizes whitespace. NLTK utilizes 179 stop words; spaCy utilizes 326.
3. Token Filtering: Stop-word removal filters out ~35–38% of tokens in NLTK and ~40–43% in spaCy.
4. Lemmatization: Both handle homographs (e.g., "Book" vs "book"), but spaCy handles irregular forms natively via its dependency pipeline.
5. NER: NLTK applies coarse labels (PERSON, GPE), while spaCy provides fine-grained extraction (MONEY, DATE, TIME, CARDINAL).
6. Limitations & Ethics: Uppercase words and slang cause false entity tags and misclassifications; pre-masking personal data ensures privacy compliance.

### Short Viva Voce Answers

1. Why is lemmatization usually more linguistically meaningful than stemming?
   
-   Lemmatization returns a valid dictionary base word (lemma) using grammar and context (e.g., "better" $\rightarrow$ "good", "went" $\rightarrow$ "go").
   
-   Stemming simply chops off suffixes using fixed rules, often creating non-words (e.g., "studies" $\rightarrow$ "studi").

2. When should stop words be retained?

- Sentiment Analysis: Words like "not" or "never" flip sentence meaning (removing "not" turns "not good" into "good").

- Translation & Question Answering: Words like "who", "where", and "to" preserve grammar and meaning.

3. How do POS tags support lemmatization?

- POS tags tell the lemmatizer whether a word is a noun, verb, or adjective so it chooses the right root form. For example, "saw" as a Verb becomes "see", while "saw" as a Noun stays "saw".

4. What causes NER errors in domain-specific text?

- Capitalization Noise: ALL-CAPS spam words ("FREE", "URGENT") get mistakenly tagged as organizations or people.

- Slang & Abbreviations: SMS shortcuts ("2morrow", "b4", "ur") and local slang ("lar", "wif") do not appear in the standard news datasets used to train NER models.